# BayesNF — k-fold sweep (VI on all 5 folds)

Runs the baseline `train_eval_vi(...)` configuration sequentially over
folds **0..4**. Per-fold artifacts are saved under unique names:

```
s3://thesis-data-ismaktam/bayesnf/runs/vi__kfold{i}__WY_h1_10__ffrk_full/
  config.json   metrics.json   preds.parquet   losses.npy   model.pkl
```

After the loop: concatenate predictions across the 5 folds into a single
OOF set, recompute CRPS / MAE / RMSE / cov on the union, plus a
sanity check that fold row counts sum to the full test set. This gives
an apples-to-apples comparison with the LGBM 5-fold OOF.

> Cost: one VI run takes ~1.5–2 h on 4×A100. Five folds ~ 8–10 h of
> continuous rental. Make sure ssh uptime and ~6 GB of local disk are
> available before launching.


## 1. Environment + imports


In [ ]:
import os
# JAX env vars must be set BEFORE the first `import jax`
os.environ.setdefault('JAX_LOG_COMPILES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.90')

# Speedup flags validated in speedup/speedup.ipynb (Triton GEMM,
# latency-hiding scheduler, async collectives). Smoke A/B matched tf32
# loss to 5 significant digits.
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_enable_triton_gemm=true '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
    '--xla_gpu_enable_async_collectives=true'
)

import sys
import gc
import time
import json
import pickle
import threading
from pathlib import Path

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
import bayesnf
from bayesnf.spatiotemporal import (
    BayesianNeuralFieldVI,
    BayesianNeuralFieldMAP,
)

# bf16 mixed precision enables full-bandwidth A100 tensor cores.
# Smoke A/B vs tf32 in notebooks/05_bayesnf/speedup/: ELBO stayed finite
# and loss matched to 5 significant digits. Main benefit: ~1/2 peak
# activation memory, which keeps batch=131072 feasible for >10-year periods.
jax.config.update('jax_default_matmul_precision', 'bfloat16')
PRECISION_NOTE = 'bfloat16 (A100 tensor cores)'

# Repo root: notebooks/05_bayesnf/kfold/<this>.ipynb -> ../../..
ROOT = Path('../../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'cwd      : {os.getcwd()}')
print(f'bayesnf  : {getattr(bayesnf, "__version__", "n/a")}')
print(f'precision: {PRECISION_NOTE}')


## 2. GPU diagnostics


In [ ]:
import subprocess

backend = jax.default_backend()
devices = jax.devices()
print(f'JAX backend : {backend}')
print(f'JAX devices : {devices}')
print(f'n local     : {jax.local_device_count()}')

if backend != 'gpu':
    print('\nWARNING: JAX running on CPU - this notebook expects GPU.')

try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    print('\nnvidia-smi:')
    print(smi.stdout)
except Exception as e:
    print(f'nvidia-smi unavailable: {e}')


## 3. Data and target config

Likelihood / target / data period. Model hyperparameters live in the
train_eval cells below.


In [ ]:
# --- Likelihood and target ---
LIKELIHOOD    = 'ZINB'
TARGET_COL    = 'rainfall_int'
NEEDS_RESCALE = True
PRECIP_SCALE  = 10

# --- Data period ---
# FOLD is bound per iteration in the sweep cell below.
YEAR_START = 2013
YEAR_END   = 2022

# --- Output ---
LOCAL_RUNS = Path('results/bayesnf/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

# --- S3 ---
S3_BUCKET    = 'thesis-data-ismaktam'
S3_RUNS_ROOT = 'bayesnf/runs'

print(f'likelihood   : {LIKELIHOOD}  (target={TARGET_COL}, rescale={NEEDS_RESCALE})')
print(f'years        : {YEAR_START}-{YEAR_END}')
print(f'local runs   : {LOCAL_RUNS}')
print(f's3 prefix    : s3://{S3_BUCKET}/{S3_RUNS_ROOT}/<name>/')


## 5. JIT-cache patch for `predict`

Same patch as in time_seasonality / geo_features. Without it `predict_bnf`
recompiles its closure for every chunk.


In [ ]:
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st
from tensorflow_probability.substrates import jax as _tfp_jax

_tfd = _tfp_jax.distributions
_LD  = bnf_models.LikelihoodDist

# --- bounded likelihood (anti-blowup for NB / ZINB) -------------------------
# NB/ZINB parameterisation in bayesnf: mean = softplus(MLP(x)),
# shape = softplus(p1). The MLP output is unbounded above, so on test rows
# with extreme features `mean` blows up to ~10^6 mm. Guard with a thin
# tanh wrapper around the raw parameters.
#
#  raw_pred  -> PRED_CAP * tanh(raw_pred / PRED_CAP)   (soft clip; for
#               |raw_pred| << PRED_CAP it is effectively linear).
#  params[1] -> SHAPE_CAP * tanh(params[1] / SHAPE_CAP) -> shape in ~[2.5e-3, 6]
#               -> total_count = 1/shape in ~[0.17, 400].
#
# Target mean range in rainfall_int units: <= 10 * 200 mm = 2000.
# PRED_CAP = 4000 (2x headroom) -> softplus(+/-4000) ~ {~0, 4000}.
PRED_CAP  = 4000.0
SHAPE_CAP = 6.0

_orig_make_likelihood = bnf_models.make_likelihood_model


def make_likelihood_bounded(params, x, mlp, mlp_template, distribution):
    if _LD(distribution) == _LD.NORMAL:
        return _orig_make_likelihood(params, x, mlp, mlp_template, distribution)

    treedef    = jax.tree_util.tree_structure(mlp_template)
    mlp_params = jax.tree_util.tree_unflatten(treedef, params[3:])
    raw_pred   = mlp.apply(mlp_params, x)
    bounded_pred = PRED_CAP * jnp.tanh(raw_pred / PRED_CAP)
    mean       = jax.nn.softplus(bounded_pred)

    bounded_p1 = SHAPE_CAP * jnp.tanh(params[1] / SHAPE_CAP)
    shape      = jax.nn.softplus(bounded_p1)
    logits     = -jnp.log(shape) - jnp.log(mean)

    if _LD(distribution) == _LD.NB:
        return _tfd.Independent(
            _tfd.NegativeBinomial(total_count=1.0 / shape, logits=logits), 1)
    elif _LD(distribution) == _LD.ZINB:
        inflated_loc_probs = 1.0 / (1.0 + jnp.exp(-params[2]))
        return _tfd.Independent(
            _tfd.ZeroInflatedNegativeBinomial(
                total_count=1.0 / shape, logits=logits,
                inflated_loc_probs=inflated_loc_probs * jnp.ones(shape=mean.shape)), 1)
    raise AssertionError(f'unknown distribution: {distribution}')


bnf_models.make_likelihood_model = make_likelihood_bounded
print(f'bounded likelihood: mean <= ~{PRED_CAP:.0f} (rainfall_int units = '
      f'{PRED_CAP/PRECIP_SCALE:.0f} mm/day),  total_count in [{1/jax.nn.softplus(SHAPE_CAP):.3f}, '
      f'{1/jax.nn.softplus(-SHAPE_CAP):.1f}]')


# --- predict JIT-cache (same as in time_seasonality / geo_features) --------
def _patched_predict(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict
print('patched BayesianNeuralFieldEstimator.predict')


## 5b. Multi-GPU predict — larger inner batchsize

`bayesnf.inference.forecast_parameters_batched` processes test data in
**chunks of 1024 rows** in a Python loop. With PRED_CHUNK = 500_000
that's ~488 sequential pmap calls per chunk. On A100/A40 a forward pass
over 1024 rows is too small to saturate the cards; the GPUs look loaded
but real utilisation is low.

Fix: same pmap(vmap(vmap)) structure as in baseline but with
`batchsize >= 16_384`. Less Python overhead, higher utilisation across
all cards, predict speedup of **5–10x** depending on instance type.

`N_DEVICES` is auto-detected via `jax.local_device_count()`, so the
notebook runs on 1x / 4x / 8x GPU without edits.

> Sanity: results are bit-equivalent to baseline (same pmap structure,
> only the inner batch size differs). To verify, run one small fold
> with both patches and `assert np.allclose(...)`.


In [ ]:
import jax
import jax.numpy as jnp
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st

N_DEVICES = jax.local_device_count()

# Per-device inner batch. Memory peak per device:
#   per_device_batch * ensemble_size * MLP_width * n_layers * dtype_bytes
# For ensemble_size=16, width=256, depth=2, float32 ~ 130 KB/row -> 16k rows ~ 2GB.
# Comfortable for 40-80GB cards; cap conservatively for smaller GPUs.
PER_DEVICE_BATCH = 16_384
PREDICT_BATCHSIZE = max(PER_DEVICE_BATCH * N_DEVICES, 16_384)

print(f'[predict-tune] N_DEVICES         = {N_DEVICES}')
print(f'[predict-tune] PER_DEVICE_BATCH  = {PER_DEVICE_BATCH:,}')
print(f'[predict-tune] PREDICT_BATCHSIZE = {PREDICT_BATCHSIZE:,}  '
      f'(was 1024 in baseline -> ~{PREDICT_BATCHSIZE//1024}x larger)')


def _patched_predict_v2(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
        batchsize=PREDICT_BATCHSIZE,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict_v2
print(f'predict re-patched with batchsize={PREDICT_BATCHSIZE:,} '
      f'across {N_DEVICES} device(s)')


## 6. Helpers: metrics, save, S3 upload, train_eval_vi / train_eval_map

One large block. Run once; the training cells below just call
`train_eval_vi(name=..., **overrides)` or `train_eval_map(...)`.

**Metrics groups.**

| Group    | Slice          | Metrics |
|----------|----------------|---------|
| Global   | all test rows  | RMSE, MAE, bias, CRPS, cov90, cov80 |
| Wet      | `y >= 0.5 mm`  | RMSE_wet, MAE_wet, CRPS_wet, cov90_wet, cov80_wet |
| Dry      | `y < 0.5 mm`   | MAE_dry, RMSE_dry, bias_dry |
| Detector | wet-classifier, threshold `predicted >= 0.4 mm`, observed `>= 0.5 mm` | TP/FP/TN/FN, accuracy, precision, recall, F1, Brier |


In [ ]:
from properscoring import crps_ensemble
import boto3

# --- thresholds ---
WET_THRESHOLD_MM   = 0.5   # y_true wet iff y >= 0.5 mm
PRED_DRY_THRESHOLD = 0.4   # model predicts dry iff mean_mm < 0.4

# --- quantiles ---
QUANTILES = (0.05, 0.1, 0.2, 0.3, 0.4, 0.50, 0.6, 0.7, 0.8, 0.9, 0.95)
QUANTILE_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
Q05_I = QUANTILE_LABELS.index(5)
Q10_I = QUANTILE_LABELS.index(10)
Q50_I = QUANTILE_LABELS.index(50)
Q90_I = QUANTILE_LABELS.index(90)
Q95_I = QUANTILE_LABELS.index(95)

PRED_CHUNK = 500_000

# --- defaults (geo features + seasonality) ---
GEO_DEFAULTS = dict(
    feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m',
                  'idw', 'gos',
                  'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
                  'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
    interactions=[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5),
                  (1, 2), (1, 3), (2, 3)],
)
SEAS_DEFAULT = dict(periods=['W', 'Y'], harmonics=[1, 10])


# ----------------------------------------------------------------------
def _metrics(y_true, mean_mm, q_arr):
    """Returns dict with global / wet / dry / detector groups."""
    lo90 = q_arr[:, Q05_I]; hi90 = q_arr[:, Q95_I]
    lo80 = q_arr[:, Q10_I]; hi80 = q_arr[:, Q90_I]

    obs_wet  = y_true  >= WET_THRESHOLD_MM
    pred_wet = mean_mm >= PRED_DRY_THRESHOLD
    tp = int(( obs_wet &  pred_wet).sum())
    tn = int((~obs_wet & ~pred_wet).sum())
    fp = int((~obs_wet &  pred_wet).sum())
    fn = int(( obs_wet & ~pred_wet).sum())
    n  = len(y_true)
    eps = 1e-12

    accuracy  = (tp + tn) / max(n, 1)
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, eps)
    brier     = float(np.mean((pred_wet.astype(np.float32) - obs_wet.astype(np.float32))**2))

    out = {
        # global
        'rmse'    : float(np.sqrt(np.mean((mean_mm - y_true) ** 2))),
        'mae'     : float(np.mean(np.abs(mean_mm - y_true))),
        'bias'    : float(np.mean(mean_mm - y_true)),
        'crps'    : float(crps_ensemble(y_true, q_arr).mean()),
        'cov90'   : float(np.mean((y_true >= lo90) & (y_true <= hi90))),
        'cov80'   : float(np.mean((y_true >= lo80) & (y_true <= hi80))),
        'n_total' : int(n),

        # detector
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'accuracy' : float(accuracy),
        'precision': float(precision),
        'recall'   : float(recall),
        'f1'       : float(f1),
        'brier'    : brier,
    }

    # wet slice
    if obs_wet.sum() > 0:
        out.update({
            'rmse_wet'  : float(np.sqrt(np.mean((mean_mm[obs_wet] - y_true[obs_wet]) ** 2))),
            'mae_wet'   : float(np.mean(np.abs(mean_mm[obs_wet] - y_true[obs_wet]))),
            'crps_wet'  : float(crps_ensemble(y_true[obs_wet], q_arr[obs_wet]).mean()),
            'cov90_wet' : float(np.mean((y_true[obs_wet] >= lo90[obs_wet]) & (y_true[obs_wet] <= hi90[obs_wet]))),
            'cov80_wet' : float(np.mean((y_true[obs_wet] >= lo80[obs_wet]) & (y_true[obs_wet] <= hi80[obs_wet]))),
            'n_wet'     : int(obs_wet.sum()),
        })
    else:
        out.update({k: float('nan') for k in
                    ('rmse_wet','mae_wet','crps_wet','cov90_wet','cov80_wet')})
        out['n_wet'] = 0

    # dry slice
    if (~obs_wet).sum() > 0:
        d = ~obs_wet
        out.update({
            'mae_dry'  : float(np.mean(np.abs(mean_mm[d] - y_true[d]))),
            'rmse_dry' : float(np.sqrt(np.mean((mean_mm[d] - y_true[d]) ** 2))),
            'bias_dry' : float(np.mean(mean_mm[d] - y_true[d])),
            'n_dry'    : int(d.sum()),
        })
    else:
        out.update({'mae_dry': float('nan'), 'rmse_dry': float('nan'),
                    'bias_dry': float('nan'), 'n_dry': 0})
    return out


def _save_run(run_name: str, model, losses, preds_df, metrics, config):
    """Persist all artifacts under LOCAL_RUNS/<run_name>/.

    Known TFP issue: `model.params_` is a `structural_tuple.StructTuple`
    whose class is created dynamically inside a function (`weakref` cache),
    so `pickle` cannot resolve it by qualified name on load. Workaround:
    flatten into a dict with explicit field names and store as plain
    numpy leaves.
    """
    sub_dir = LOCAL_RUNS / run_name
    sub_dir.mkdir(parents=True, exist_ok=True)

    np.save(sub_dir / 'losses.npy', np.asarray(losses))
    with open(sub_dir / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    with open(sub_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    preds_df.to_parquet(sub_dir / 'preds.parquet', index=False)

    # Flatten StructTuple -> dict.
    p = model.params_
    params_dict = {name: np.asarray(val) for name, val in zip(p._fields, p)}

    # Store every constructor argument so the model can be rebuilt on load.
    init_kwargs = {
        'width': model.width, 'depth': model.depth,
        'freq': model.freq, 'timetype': model.timetype,
        'seasonality_periods': list(model.seasonality_periods),
        'num_seasonal_harmonics': list(model.num_seasonal_harmonics),
        'feature_cols': list(model.feature_cols),
        'target_col':  model.target_col,
        'standardize': list(model.standardize),
        'observation_model': model.observation_model,
        'interactions': [list(t) for t in model.interactions]
                         if model.interactions is not None else None,
    }

    # data_handler scalers (mean / std for standardize) — needed at predict time.
    dh = model.data_handler
    standardize_ = {
        'mean': np.asarray(dh.standardize_['mean'])  if hasattr(dh, 'standardize_') and dh.standardize_ else None,
        'std' : np.asarray(dh.standardize_['std'])   if hasattr(dh, 'standardize_') and dh.standardize_ else None,
    }

    payload = {
        'params_dict' : params_dict,
        'param_fields': list(p._fields),
        'init_kwargs' : init_kwargs,
        'standardize_': standardize_,
        'inference'   : config.get('inference'),
    }
    with open(sub_dir / 'model.pkl', 'wb') as f:
        pickle.dump(payload, f)
    print(f'  saved {len(params_dict)} param fields + meta to {sub_dir / "model.pkl"}')
    return sub_dir


def _load_run_model(run_name: str):
    """Counterpart to _save_run: return an estimator ready to predict.

    Not strictly needed in the baseline notebook (it reads preds.parquet
    directly), but useful for grid predictions later.
    """
    sub_dir = LOCAL_RUNS / run_name
    with open(sub_dir / 'model.pkl', 'rb') as f:
        payload = pickle.load(f)

    from tensorflow_probability.python.internal import structural_tuple
    inference = payload['inference']
    cls = bnf_st.BayesianNeuralFieldVI if inference == 'vi' else bnf_st.BayesianNeuralFieldMAP
    model = cls(**payload['init_kwargs'])

    # Rebuild StructTuple from dict
    StructT = structural_tuple.structtuple(payload['param_fields'])
    model.params_ = StructT(*(payload['params_dict'][k] for k in payload['param_fields']))

    # Restore scalers
    std = payload.get('standardize_') or {}
    if std.get('mean') is not None and std.get('std') is not None:
        model.data_handler.standardize_ = {
            'mean': std['mean'], 'std': std['std'],
        }
    return model


def _upload_run_s3(run_name: str) -> None:
    """Idempotently upload LOCAL_RUNS/<run_name>/ to s3://bucket/bayesnf/runs/<run_name>/."""
    src_dir = LOCAL_RUNS / run_name
    if not src_dir.exists():
        print(f'  {src_dir} does not exist, skip upload')
        return
    s3 = boto3.client('s3')
    for f in sorted(src_dir.iterdir()):
        if not f.is_file():
            continue
        key = f'{S3_RUNS_ROOT}/{run_name}/{f.name}'
        s3.upload_file(str(f), S3_BUCKET, key)
        size_mb = f.stat().st_size / 1e6
        print(f'  s3://{S3_BUCKET}/{key}  ({size_mb:.1f} MB)')


# ----------------------------------------------------------------------
def _build_predict_metrics(model, df_test_sub, name):
    """Run chunked predict, compute metrics, return (mean_per_point, q_arr, metrics_dict, preds_df, predict_time_s)."""
    means_chunks, q_chunks = [], []
    t0 = time.time()
    n = len(df_test_sub)
    for i in range(0, n, PRED_CHUNK):
        j = min(i + PRED_CHUNK, n)
        m, q = model.predict(df_test_sub.iloc[i:j], quantiles=QUANTILES, approximate_quantiles=True)
        means_chunks.append(np.asarray(m))
        q_chunks.append(np.asarray(q))
    gc.collect()
    predict_time_s = time.time() - t0
    print(f'  predict {n:,} rows in {predict_time_s:.0f}s')

    means_raw = np.concatenate(means_chunks, axis=-1)
    q_raw     = np.concatenate(q_chunks, axis=-1)
    mean_per_point = means_raw.reshape(-1, n).mean(axis=0)
    q_flat = q_raw.reshape(q_raw.shape[0], -1, n).mean(axis=1)

    if NEEDS_RESCALE:
        mean_per_point = mean_per_point / PRECIP_SCALE
        q_flat = q_flat / PRECIP_SCALE

    # --- diagnostics + safety clip ---------------------------------------
    n_bad_mean = int((~np.isfinite(mean_per_point)).sum())
    n_bad_q    = int((~np.isfinite(q_flat)).sum())
    abs_mean   = np.abs(mean_per_point)
    pct = np.nanpercentile(abs_mean, [50, 95, 99, 99.9])
    print(f'  diag |mean_mm|: p50={pct[0]:.2f}  p95={pct[1]:.2f}  '
          f'p99={pct[2]:.2f}  p99.9={pct[3]:.2f}  max={np.nanmax(abs_mean):.2f}')
    print(f'  diag non-finite: mean={n_bad_mean} ({n_bad_mean/n*100:.3f}%) '
          f'q={n_bad_q} ({n_bad_q/q_flat.size*100:.3f}%)')

    # Safety-net clip: 250 mm/day absolute max for station data. The world
    # daily record is ~1825 mm, an extreme outlier; realistic max for
    # Germany is <= 200 mm. Mean is clipped harder than quantiles so
    # calibration isn't distorted.
    HARD_CAP_MEAN_MM = 250.0
    HARD_CAP_Q_MM    = 500.0
    n_clip_mean = int((mean_per_point > HARD_CAP_MEAN_MM).sum())
    n_clip_q    = int((q_flat > HARD_CAP_Q_MM).sum())
    if n_clip_mean or n_clip_q:
        print(f'  clipping: mean>{HARD_CAP_MEAN_MM}mm n={n_clip_mean}, '
              f'q>{HARD_CAP_Q_MM}mm n={n_clip_q}')
    mean_per_point = np.clip(np.nan_to_num(mean_per_point, nan=0.0,
                                           posinf=HARD_CAP_MEAN_MM, neginf=0.0),
                             0.0, HARD_CAP_MEAN_MM)
    q_flat = np.clip(np.nan_to_num(q_flat, nan=0.0,
                                    posinf=HARD_CAP_Q_MM, neginf=0.0),
                     0.0, HARD_CAP_Q_MM)

    y_true = df_test_sub['rainfall'].to_numpy()
    q_arr = np.stack(list(q_flat), axis=1)
    q_arr = np.sort(q_arr, axis=1)

    metrics = _metrics(y_true, mean_per_point, q_arr)
    metrics.update({
        'name': name, 'n_quantiles': len(QUANTILES),
        'predict_time_s': float(predict_time_s),
        'n_nonfinite_mean': n_bad_mean,
        'n_nonfinite_q'   : n_bad_q,
        'n_clipped_mean'  : n_clip_mean,
        'n_clipped_q'     : n_clip_q,
        'p99_abs_mean_mm' : float(pct[2]),
        'p999_abs_mean_mm': float(pct[3]),
    })

    preds_df = df_test_sub[['station_id', 'datetime']].copy()
    preds_df['observed_mm'] = y_true
    preds_df['mean_mm']     = mean_per_point
    for lbl, qv in zip(QUANTILE_LABELS, q_flat):
        preds_df[f'q{lbl:02d}'] = qv

    return mean_per_point, q_arr, metrics, preds_df


def _print_summary(metrics):
    m = metrics
    print(f'  CRPS     = {m["crps"]:.4f}    CRPS_wet = {m["crps_wet"]:.4f}')
    print(f'  MAE      = {m["mae"]:.4f}    MAE_wet  = {m["mae_wet"]:.4f}    MAE_dry  = {m["mae_dry"]:.4f}')
    print(f'  RMSE     = {m["rmse"]:.4f}   RMSE_wet = {m["rmse_wet"]:.4f}   RMSE_dry = {m["rmse_dry"]:.4f}')
    print(f'  bias     = {m["bias"]:+.4f}  cov90    = {m["cov90"]:.3f}      cov80    = {m["cov80"]:.3f}')
    print(f'  detector: acc={m["accuracy"]:.3f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}  Brier={m["brier"]:.4f}')
    print(f'            TP={m["tp"]:,}  FP={m["fp"]:,}  TN={m["tn"]:,}  FN={m["fn"]:,}')


def _heartbeat_start(name):
    stop = threading.Event()
    def _beat():
        t0 = time.time()
        while not stop.wait(20.0):
            print(f'    [{name}] still training ({time.time()-t0:.0f}s)', flush=True)
    thr = threading.Thread(target=_beat, daemon=True); thr.start()
    return stop, thr


# ----------------------------------------------------------------------
def train_eval_vi(
    name: str,
    *,
    num_epochs: int,
    batch_size: int,
    ensemble_size: int,
    learning_rate: float,
    kl_weight: float,
    sample_size_posterior: int,
    width: int = 256,
    depth: int = 2,
    seed: int = 0,
    seasonality_periods=None,
    num_seasonal_harmonics=None,
    feature_cols=None,
    interactions=None,
) -> dict:
    seasonality_periods    = seasonality_periods    or SEAS_DEFAULT['periods']
    num_seasonal_harmonics = num_seasonal_harmonics or SEAS_DEFAULT['harmonics']
    feature_cols           = feature_cols           or GEO_DEFAULTS['feature_cols']
    interactions           = interactions           or GEO_DEFAULTS['interactions']

    assert len(seasonality_periods) == len(num_seasonal_harmonics)
    assert feature_cols[0] == 'datetime'
    for i, j in interactions:
        assert 0 <= i < len(feature_cols) and 0 <= j < len(feature_cols)

    print(f'\n{"="*78}')
    print(f'  [VI] {name}')
    print(f'       seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'       features ({len(feature_cols)}): {feature_cols}')
    print(f'       width={width} depth={depth}  epochs={num_epochs}  batch={batch_size}')
    print(f'       ensemble={ensemble_size}  lr={learning_rate}  kl={kl_weight}  sample_post={sample_size_posterior}')
    print(f'{"="*78}')

    standardize_cols = [c for c in feature_cols if c != 'datetime']
    model = BayesianNeuralFieldVI(
        width=width, depth=depth,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    stop, thr = _heartbeat_start(name)
    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(seed),
            num_epochs=num_epochs,
            ensemble_size=ensemble_size,
            learning_rate=learning_rate,
            batch_size=batch_size,
            kl_weight=kl_weight,
            sample_size_posterior=sample_size_posterior,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    _, _, metrics, preds_df = _build_predict_metrics(model, df_test_sub, name)
    metrics.update({
        'inference'             : 'vi',
        'seasonality_periods'   : seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols'          : feature_cols,
        'interactions'          : interactions,
        'n_features'            : len(feature_cols),
        'fit_time_s'            : float(fit_time_s),
    })

    config = {
        'inference': 'vi', 'name': name,
        'width': width, 'depth': depth,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'ensemble_size': ensemble_size, 'learning_rate': learning_rate,
        'kl_weight': kl_weight, 'sample_size_posterior': sample_size_posterior,
        'seed': seed,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols, 'interactions': interactions,
        'likelihood': LIKELIHOOD, 'target_col': TARGET_COL,
        'fold': FOLD, 'year_start': YEAR_START, 'year_end': YEAR_END,
    }
    _save_run(name, model, losses_arr, preds_df, metrics, config)
    _upload_run_s3(name)
    _print_summary(metrics)

    del model
    jax.clear_caches(); gc.collect()
    return metrics


def train_eval_map(
    name: str,
    *,
    num_epochs: int,
    batch_size: int,
    ensemble_size: int,
    learning_rate: float,
    num_splits: int = 1,
    width: int = 256,
    depth: int = 2,
    seed: int = 0,
    seasonality_periods=None,
    num_seasonal_harmonics=None,
    feature_cols=None,
    interactions=None,
) -> dict:
    seasonality_periods    = seasonality_periods    or SEAS_DEFAULT['periods']
    num_seasonal_harmonics = num_seasonal_harmonics or SEAS_DEFAULT['harmonics']
    feature_cols           = feature_cols           or GEO_DEFAULTS['feature_cols']
    interactions           = interactions           or GEO_DEFAULTS['interactions']

    assert len(seasonality_periods) == len(num_seasonal_harmonics)
    assert feature_cols[0] == 'datetime'

    print(f'\n{"="*78}')
    print(f'  [MAP] {name}')
    print(f'        seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'        features ({len(feature_cols)}): {feature_cols}')
    print(f'        width={width} depth={depth}  epochs={num_epochs}  batch={batch_size}')
    print(f'        ensemble={ensemble_size}  lr={learning_rate}  splits={num_splits}')
    print(f'{"="*78}')

    standardize_cols = [c for c in feature_cols if c != 'datetime']
    model = BayesianNeuralFieldMAP(
        width=width, depth=depth,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    stop, thr = _heartbeat_start(name)
    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(seed),
            num_epochs=num_epochs,
            ensemble_size=ensemble_size,
            learning_rate=learning_rate,
            batch_size=batch_size,
            num_splits=num_splits,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    _, _, metrics, preds_df = _build_predict_metrics(model, df_test_sub, name)
    metrics.update({
        'inference'             : 'map',
        'seasonality_periods'   : seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols'          : feature_cols,
        'interactions'          : interactions,
        'n_features'            : len(feature_cols),
        'fit_time_s'            : float(fit_time_s),
    })

    config = {
        'inference': 'map', 'name': name,
        'width': width, 'depth': depth,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'ensemble_size': ensemble_size, 'learning_rate': learning_rate,
        'num_splits': num_splits, 'seed': seed,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols, 'interactions': interactions,
        'likelihood': LIKELIHOOD, 'target_col': TARGET_COL,
        'fold': FOLD, 'year_start': YEAR_START, 'year_end': YEAR_END,
    }
    _save_run(name, model, losses_arr, preds_df, metrics, config)
    _upload_run_s3(name)
    _print_summary(metrics)

    del model
    jax.clear_caches(); gc.collect()
    return metrics


print('helpers ready')
print('  GEO_DEFAULTS : n_features =', len(GEO_DEFAULTS['feature_cols']),
      ', n_interactions =', len(GEO_DEFAULTS['interactions']))
print('  SEAS_DEFAULT :', SEAS_DEFAULT)
print('  thresholds   : observed wet >=', WET_THRESHOLD_MM, 'mm,  predicted wet >=', PRED_DRY_THRESHOLD, 'mm')


## 7. K-fold sweep — configuration

`FOLDS` defaults to all 5 indices. To run a subset (e.g. resume after a
crash) keep only the indices needed.

`RUN_TAG` is part of each run name; change it only to start a parallel
experiment branch without overwriting the previous one.


In [ ]:
FOLDS         = [0, 1, 2, 3, 4]
RUN_TAG       = 'kfold'      # -> name = f'vi__{RUN_TAG}{i}__WY_h1_10__ffrk_full'
RUN_MAP       = False        # True -> additionally run MAP on each fold
SKIP_EXISTING = True         # if metrics for the name are already in S3, skip

# Hyperparameters — same defaults as baseline section 7. Edit here once and
# they apply to all folds.
VI_HPARAMS = dict(
    num_epochs            = 100,
    batch_size            = 131072,
    ensemble_size         = 16,
    learning_rate         = 0.005,
    kl_weight             = 0.1,
    sample_size_posterior = 5,
    width = 256, depth = 2, seed = 0,
)
MAP_HPARAMS = dict(
    num_epochs    = 2000,
    batch_size    = 131072,
    ensemble_size = 16,
    learning_rate = 0.005,
    num_splits    = 1,
    width = 256, depth = 2, seed = 0,
)

print(f'  folds       : {FOLDS}')
print(f'  run pattern : vi__{RUN_TAG}{{i}}__WY_h1_10__ffrk_full')
print(f'  skip done   : {SKIP_EXISTING}')
print(f'  run MAP too : {RUN_MAP}')


## 8. K-fold sweep — main loop

For each fold:

1. **Cache check.** If `SKIP_EXISTING=True` and the run's metrics are
   already in S3, the fold is skipped (resume-friendly).
2. **Data load.** Pull `bayesnf_fold{i}_{train,test}.parquet` from S3,
   filtered by `YEAR_START..YEAR_END`.
3. **Train + predict + save + upload.** Call `train_eval_vi(...)` (and
   optionally `train_eval_map`). Between folds: `jax.clear_caches() + gc.collect()`.
4. **Logging.** Per-fold metrics are appended to `kfold_metrics`. After
   the loop the table is printed and saved as
   `results/bayesnf/kfold_summary.json`.

`df_train`/`df_test` and `FOLD` are written to the global namespace
before each call; `train_eval_vi` picks them up via Python closure
lookup.


In [ ]:
import boto3
from botocore.exceptions import ClientError

s3_client = boto3.client('s3')

def _s3_has_metrics(run_name: str) -> bool:
    key = f'{S3_RUNS_ROOT}/{run_name}/metrics.json'
    try:
        s3_client.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except ClientError:
        return False

date_filter = [
    ('datetime', '>=', pd.Timestamp(f'{YEAR_START}-01-01')),
    ('datetime', '<=', pd.Timestamp(f'{YEAR_END}-12-31')),
]
ALL_GEO_COLS = [
    'datetime', 'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
    'idw', 'gos',
    'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
    'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
]
KEEP_COLS = ALL_GEO_COLS + ['rainfall', 'rainfall_int', 'station_id']

DATA_S3_DIR = 'bayesnf/data'
DATA_DIR    = Path('results/bayesnf/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def _ensure_local(local_path, s3_key):
    if local_path.exists():
        return
    print(f'    downloading s3://{S3_BUCKET}/{s3_key} ...')
    s3_client.download_file(S3_BUCKET, s3_key, str(local_path))

kfold_metrics = []
t_sweep = time.time()

for fold in FOLDS:
    print(f'\n{"#"*78}')
    print(f'   FOLD {fold} of {FOLDS}')
    print(f'{"#"*78}')

    name_vi  = f'vi__{RUN_TAG}{fold}__WY_h1_10__ffrk_full'
    name_map = f'map__{RUN_TAG}{fold}__WY_h1_10__ffrk_full'

    # Per-inference skip: each inference is checked independently. This
    # lets us, e.g., finish MAP on all folds without recomputing VI runs
    # that are already in S3 (or vice versa).
    skip_vi  = SKIP_EXISTING and _s3_has_metrics(name_vi)
    skip_map = SKIP_EXISTING and _s3_has_metrics(name_map)
    need_vi  = not skip_vi
    need_map = RUN_MAP and not skip_map

    def _append_cached(run_name, inference):
        cache_dir = LOCAL_RUNS / run_name
        if (cache_dir / 'metrics.json').exists():
            m = json.loads((cache_dir / 'metrics.json').read_text())
            kfold_metrics.append({**m, 'fold': fold, 'inference': inference})

    if not need_vi and not need_map:
        msg_parts = [name_vi] + ([name_map] if RUN_MAP else [])
        print(f'   S3 already has metrics for {", ".join(msg_parts)} — '
              f'skipping (set SKIP_EXISTING=False to force re-run)')
        _append_cached(name_vi, 'vi')
        if RUN_MAP:
            _append_cached(name_map, 'map')
        continue

    # At least one inference to train -> load data
    train_path = DATA_DIR / f'bayesnf_fold{fold}_train.parquet'
    test_path  = DATA_DIR / f'bayesnf_fold{fold}_test.parquet'
    _ensure_local(train_path, f'{DATA_S3_DIR}/bayesnf_fold{fold}_train.parquet')
    _ensure_local(test_path,  f'{DATA_S3_DIR}/bayesnf_fold{fold}_test.parquet')

    df_train = pd.read_parquet(train_path, filters=date_filter)[KEEP_COLS].copy()
    df_test  = pd.read_parquet(test_path,  filters=date_filter)[KEEP_COLS].copy()
    df_train['datetime'] = pd.to_datetime(df_train['datetime'])
    df_test ['datetime'] = pd.to_datetime(df_test ['datetime'])

    print(f'   fold {fold} train : {len(df_train):>10,} rows / '
          f'{df_train["station_id"].nunique()} stations')
    print(f'   fold {fold} test  : {len(df_test):>10,} rows / '
          f'{df_test ["station_id"].nunique()} stations')
    print(f'   plan      : VI={"train" if need_vi else "skip (cached)"}, '
          f'MAP={"train" if need_map else ("skip (cached)" if RUN_MAP else "off")}')

    # Rebind FOLD so train_eval_* writes the correct fold into config.json
    FOLD = fold

    # --- VI ---
    if need_vi:
        m_vi = train_eval_vi(name=name_vi, **VI_HPARAMS)
        kfold_metrics.append({**m_vi, 'fold': fold, 'inference': 'vi'})
    elif skip_vi:
        _append_cached(name_vi, 'vi')

    # --- MAP (optional) ---
    if need_map:
        m_map = train_eval_map(name=name_map, **MAP_HPARAMS)
        kfold_metrics.append({**m_map, 'fold': fold, 'inference': 'map'})
    elif skip_map and RUN_MAP:
        _append_cached(name_map, 'map')

    # Free GPU memory between folds (datasets are 1+ GB each)
    del df_train, df_test
    jax.clear_caches()
    gc.collect()

elapsed = time.time() - t_sweep
print(f'\n{"="*78}')
print(f'  K-fold sweep done in {elapsed/60:.1f} min  '
      f'({len(kfold_metrics)} runs)')
print(f'{"="*78}')


## 9. OOF aggregation — concat across folds, recompute metrics

Download `preds.parquet` from each fold run and concatenate by rows.
That is the full OOF set (folds are disjoint by `station_id` by
construction). Headline metrics are recomputed on the union and saved
to `results/bayesnf/kfold_oof.parquet`.

Also: a per-fold table with CRPS / MAE / RMSE / cov90 for copy-paste
into the thesis comparison section.


In [ ]:
import boto3
from properscoring import crps_ensemble

INFERENCE = 'vi'   # switch to 'map' if RUN_MAP=True was used

s3 = boto3.client('s3')
oof_chunks = []
per_fold_rows = []

for fold in FOLDS:
    name = f'{INFERENCE}__{RUN_TAG}{fold}__WY_h1_10__ffrk_full'
    run_dir = LOCAL_RUNS / name
    pred_path = run_dir / 'preds.parquet'
    metr_path = run_dir / 'metrics.json'

    # Re-pull from S3 if local cache missing (e.g. after instance restart)
    for fname in ('preds.parquet', 'metrics.json'):
        p = run_dir / fname
        if not p.exists():
            run_dir.mkdir(parents=True, exist_ok=True)
            s3.download_file(S3_BUCKET, f'{S3_RUNS_ROOT}/{name}/{fname}', str(p))

    df = pd.read_parquet(pred_path)
    df['fold'] = fold
    oof_chunks.append(df)

    m = json.loads(metr_path.read_text())
    per_fold_rows.append({
        'fold'    : fold,
        'crps'    : m['crps'],
        'crps_wet': m.get('crps_wet', float('nan')),
        'mae'     : m['mae'],
        'rmse'    : m['rmse'],
        'bias'    : m['bias'],
        'cov90'   : m['cov90'],
        'cov80'   : m['cov80'],
        'n_total' : m['n_total'],
        'n_wet'   : m.get('n_wet', None),
        'fit_s'   : m.get('fit_time_s'),
        'predict_s': m.get('predict_time_s'),
    })

oof = pd.concat(oof_chunks, ignore_index=True)
print(f'OOF rows           : {len(oof):,}')
print(f'OOF unique stations: {oof["station_id"].nunique():,}')
print(f'date range         : {oof["datetime"].min()} -> {oof["datetime"].max()}')

# Sanity check: each station should appear in exactly one fold
mult = oof.groupby('station_id')['fold'].nunique()
overlap = (mult > 1).sum()
print(f'stations in >1 fold: {overlap}  (expected 0 for clean k-fold)')

# Per-fold table
df_per_fold = pd.DataFrame(per_fold_rows).set_index('fold')
print('\n=== per-fold metrics ===')
with pd.option_context('display.float_format', '{:.4f}'.format):
    print(df_per_fold[['crps','crps_wet','mae','rmse','bias','cov90','cov80','n_total']].to_string())
print(f'\n=== fold means ===')
with pd.option_context('display.float_format', '{:.4f}'.format):
    print(df_per_fold[['crps','crps_wet','mae','rmse','bias','cov90','cov80']].mean().to_string())

# Global OOF metrics (recomputed on the union)
QUANTILE_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
q_cols = [f'q{l:02d}' for l in QUANTILE_LABELS]
y      = oof['observed_mm'].to_numpy()
mean   = oof['mean_mm'].to_numpy()
Q      = oof[q_cols].to_numpy()

oof_crps = float(crps_ensemble(y, Q).mean())
oof_mae  = float(np.mean(np.abs(mean - y)))
oof_rmse = float(np.sqrt(np.mean((mean - y) ** 2)))
oof_bias = float(np.mean(mean - y))
oof_c90  = float(np.mean((y >= Q[:, q_cols.index('q05')]) & (y <= Q[:, q_cols.index('q95')])))
oof_c80  = float(np.mean((y >= Q[:, q_cols.index('q10')]) & (y <= Q[:, q_cols.index('q90')])))

wet_mask = y >= 0.5
oof_crps_wet = float(crps_ensemble(y[wet_mask], Q[wet_mask]).mean()) if wet_mask.any() else float('nan')

print(f'\n=== GLOBAL OOF (concat across {len(FOLDS)} folds) ===')
print(f'  CRPS     : {oof_crps:.4f}      CRPS_wet : {oof_crps_wet:.4f}')
print(f'  MAE      : {oof_mae:.4f}')
print(f'  RMSE     : {oof_rmse:.4f}')
print(f'  bias     : {oof_bias:+.4f}')
print(f'  cov90    : {oof_c90:.4f}      cov80    : {oof_c80:.4f}')

# Persist OOF predictions + summary for downstream uncertainty / comparison
out_pq   = Path('results/bayesnf/kfold_oof.parquet')
out_json = Path('results/bayesnf/kfold_summary.json')
out_pq.parent.mkdir(parents=True, exist_ok=True)
oof.to_parquet(out_pq, index=False)
summary = {
    'inference': INFERENCE,
    'run_tag': RUN_TAG,
    'folds': FOLDS,
    'global_oof': {
        'crps': oof_crps, 'crps_wet': oof_crps_wet,
        'mae': oof_mae, 'rmse': oof_rmse, 'bias': oof_bias,
        'cov90': oof_c90, 'cov80': oof_c80,
        'n_total': int(len(y)), 'n_wet': int(wet_mask.sum()),
    },
    'per_fold': df_per_fold.reset_index().to_dict(orient='records'),
}
out_json.write_text(json.dumps(summary, indent=2))
print(f'\nSaved -> {out_pq}  ({out_pq.stat().st_size/1024/1024:.1f} MB)')
print(f'Saved -> {out_json}')

# Upload to S3 for downstream notebooks
s3.upload_file(str(out_pq),   S3_BUCKET, f'bayesnf/kfold_oof_{INFERENCE}.parquet')
s3.upload_file(str(out_json), S3_BUCKET, f'bayesnf/kfold_summary_{INFERENCE}.json')
print(f'Uploaded -> s3://{S3_BUCKET}/bayesnf/kfold_oof_{INFERENCE}.parquet')
